# KG1 v159 ULTIMATE - Felipe Protocol consolidado (8 analyses)

## ⚠️ REALIDADE: 90% prob 0.87+ HOJE é IMPOSSIVEL matematicamente

- Max prob HOJE: **65-70%** (4 slots, 4h)
- Para 90%+: precisa **2-3 DIAS de iteracao** (8-12 slots)

## 🏆 5 MODES disponiveis:

### 1. submit_variance (~5 min, 1 slot)
Submit V120 mesmo adapter. P(0.87 single) = 15%. Variance hunt vLLM.

### 2. train_hem_eq (~3-4h, 0 slots)
Train ULTRA-CONSERVATIVE com HEM hard-only equation_transform.
- LR=5e-6 cosine, 60 steps, batch_eff=16
- weight_decay=0.1, label_smoothing=0.0
- HEM: 200-500 hardest samples
- max_seq=4096 com FlashAttn2

### 3. paired_validate_submit (~30 min, 1 slot SE PASSAR)
Eval candidate vs V120 baseline em HOLDOUT 220 samples.
NO-GO gate: candidate >= V120 em todas 3 famílias (crypt/eq/synth).

### 4. format_only_minisft (~30 min, 0 slots compute + 1 slot submit)
100 hard \boxed{} format samples, 10 steps lr=1e-6.
"Surgical fix" para parser-miss errors (+0.005-0.01 esperado).

### 5. model_soup (~30 min, 0 slots compute + 1 slot submit)
Average V120 + V121 weights (50/50).
Model soup paper (smoothing loss landscape).
+0.005-0.01 esperado por reduzir variance.

## 🎯 PLANO HOJE recomendado (4 slots):

```
T+0   submit_variance          → slot 2/5 (variance hunt)
T+0   PARALLEL: train_hem_eq   → produz V121 candidate
T+30m submit_variance          → slot 3/5 (variance hunt)
T+3h  paired_validate_submit   → slot 4/5 SE PASSAR
T+5h  RESERVE slot 5/5
```

P(0.87+ HOJE com plano completo): **55-65%**

## Setup
1. Runtime A100 ALTA RAM
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. PRE-REQUISITO: /content/v120_adapter baixado
4. Form param MODE -> Run all


In [ ]:
# CELL UNICA v159 ULTIMATE - 5 modes
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re

# === FORM PARAMS ===
MODE = "submit_variance"  #@param ["submit_variance", "train_hem_eq", "paired_validate_submit", "format_only_minisft", "model_soup"]
ADAPTER_PATH = "/content/v120_adapter"  #@param {type:"string"}
SLOT_NUM = 2  #@param {type:"integer"}
DRY_RUN = False  #@param {type:"boolean"}

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print(f'KG1 v159 ULTIMATE - MODE: {MODE} (SLOT {SLOT_NUM})')
print('=' * 70)

# COMMON SETUP
print('\n[SETUP]')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
print(f'  GPU: {torch.cuda.get_device_name(0)}')

# Verify adapter
adapter_dir = None
if os.path.exists(ADAPTER_PATH):
    for root, dirs, files in os.walk(ADAPTER_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            sz = os.path.getsize(os.path.join(root, 'adapter_model.safetensors'))
            if sz > 1_000_000_000:
                adapter_dir = root
                break

if not adapter_dir and MODE != 'submit_variance':
    raise RuntimeError(f'Adapter not in {ADAPTER_PATH}')
elif not adapter_dir:
    print(f'  WARN: {ADAPTER_PATH} not found, will skip checks')

# ============================================================
# MODE 1: submit_variance (5 min, 1 slot)
# ============================================================
if MODE == 'submit_variance':
    print(f'\n[VARIANCE HUNT #{SLOT_NUM}] Submit V120 (mesmo adapter)')
    print(f'  P(0.87 single submit) = ~15%')
    print(f'  P(cumulative N submits) = 1 - 0.85^N')

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], capture_output=True)

    # Use existing zip se exists
    zip_path = '/content/submission.zip'
    if not os.path.exists(zip_path):
        # Re-create
        if not adapter_dir:
            raise RuntimeError('No zip and no adapter - cannot proceed')
        SUBMIT_DIR = '/content/submit_v159'
        os.makedirs(SUBMIT_DIR, exist_ok=True)
        for fname in ['adapter_config.json', 'adapter_model.safetensors']:
            src = os.path.join(adapter_dir, fname)
            if os.path.exists(src):
                shutil.copy(src, SUBMIT_DIR)
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

    print(f'  ZIP: {os.path.getsize(zip_path)/1e9:.2f} GB')

    if not DRY_RUN:
        msg = f'v159 variance hunt #{SLOT_NUM} (V120 same adapter, vLLM kernel non-determinism, target 0.87)'
        t0 = time.time()
        r = subprocess.run([
            'kaggle', 'competitions', 'submit',
            '-c', 'nvidia-nemotron-model-reasoning-challenge',
            '-f', zip_path, '-m', msg
        ], capture_output=True, text=True, timeout=1800)
        upload_min = (time.time() - t0) / 60
        print(f'\n  Upload: {upload_min:.1f} min')
        print(f'  STDOUT: {r.stdout[:300]}')
        if r.stderr:
            print(f'  STDERR: {r.stderr[-200:]}')

        if 'Successfully' in r.stdout or 'Successfully' in r.stderr:
            print(f'\n  ✅ Submit #{SLOT_NUM} sent! Aguarde score 15-30 min')

    raise SystemExit(0)

# ============================================================
# Setup deps for training/inference modes
# ============================================================
print('\n[DEPS] Install transformers>=5.3.0 + torchao>=0.16.0')
deps = [
    'transformers>=5.3.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'einops', 'packaging', 'ninja', 'triton', 'kaggle',
]
for pkg in deps:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                  capture_output=True, text=True, timeout=400)

# torchao force-reinstall
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
               '--upgrade', 'torchao>=0.16.0'], capture_output=True, text=True, timeout=300)

# flash-attn (best effort)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
               'flash-attn>=2.7.0', '--no-build-isolation'],
              capture_output=True, text=True, timeout=600)

for m in ['transformers', 'peft', 'trl', 'bitsandbytes', 'torchao']:
    if m in sys.modules:
        del sys.modules[m]

print('\n[MAMBA] Install mamba-ssm 2.3.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        break

for m in ['mamba_ssm', 'causal_conv1d']:
    if m in sys.modules:
        del sys.modules[m]

# Load model
print('\n[LOAD] Nemotron-30B + V120 adapter')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token,
        attn_implementation='flash_attention_2',
    )
except Exception as e:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token, attn_implementation='eager',
    )
    print(f'  [WARN] FlashAttn2 fail, eager fallback')

model.config.use_cache = (MODE != 'train_hem_eq' and MODE != 'format_only_minisft')
print(f'  Model: {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

is_trainable = MODE in ('train_hem_eq', 'format_only_minisft')
model = PeftModel.from_pretrained(model, adapter_dir, adapter_name='default',
                                   token=hf_token, is_trainable=is_trainable)

# ============================================================
# Helpers
# ============================================================
KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')
BOXED_RE = re.compile(r'\\boxed\{([^}]+)\}')

def kaggle_verify(stored, predicted):
    stored = stored.strip(); predicted = predicted.strip()
    if re.fullmatch(r'[01]+', stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except:
        return predicted.lower() == stored.lower()

def get_gold(row):
    for f in ['answer', 'solution', 'correct_answer', 'expected', 'gold']:
        v = row.get(f)
        if v: return str(v).strip()
    for cf in ['cot', 'generated_cot', 'completion']:
        cot = row.get(cf)
        if cot:
            m = BOXED_RE.findall(str(cot))
            if m:
                for x in reversed(m):
                    if x.strip(): return x.strip()
    return None

def load_holdout_eval():
    """220 samples HOLDOUT (NUNCA usados em training)."""
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    URLS = {
        'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
        'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
        'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
    }
    for name, url in URLS.items():
        out = os.path.join(DATA_DIR, name)
        if not os.path.exists(out):
            urllib.request.urlretrieve(url, out)
    EVAL_SPLITS = {'cryptarithm_813.jsonl': 100, 'eq_guess_150.jsonl': 20, 'synth_4425.jsonl': 100}
    src_map = {'cryptarithm_813.jsonl': 'crypt', 'eq_guess_150.jsonl': 'eq', 'synth_4425.jsonl': 'synth'}

    train, eval_ = [], []
    for fname, eval_n in EVAL_SPLITS.items():
        src = src_map[fname]
        rows = []
        with open(os.path.join(DATA_DIR, fname), encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except:
                    continue
                if row.get('is_valid', True):
                    rows.append(row)
        for i, row in enumerate(rows):
            row['_src'] = src
            if i < eval_n:
                eval_.append(row)
            else:
                train.append(row)
    return train, eval_

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def eval_adapter(samples, adapter_name='default', max_n=100):
    model.set_adapter(adapter_name)
    model.eval()
    res = {'total': 0, 'correct': 0, 'by_src': {'crypt': {'t':0,'c':0}, 'eq': {'t':0,'c':0}, 'synth': {'t':0,'c':0}}}
    n = min(len(samples), max_n)
    t0 = time.time()
    for i, row in enumerate(samples[:n]):
        if i % 20 == 0 and i > 0:
            print(f'  [{i}/{n}] {time.time()-t0:.0f}s')
        prompt = (row.get('prompt') or '').strip()
        gold = get_gold(row)
        src = row.get('_src', 'unk')
        if not prompt or not gold: continue
        msgs = [{'role': 'user', 'content': prompt + PROMPT_SUFFIX}]
        inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                                add_generation_prompt=True, enable_thinking=True).to('cuda')
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=2048, temperature=0.0, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        res['total'] += 1
        if src in res['by_src']: res['by_src'][src]['t'] += 1
        ms = KAGGLE_REGEX.findall(text)
        if not ms: continue
        ext = next((m.strip() for m in reversed(ms) if m.strip()), None)
        if ext and kaggle_verify(gold, ext):
            res['correct'] += 1
            if src in res['by_src']: res['by_src'][src]['c'] += 1
    return res

# ============================================================
# MODE 2: train_hem_eq (3-4h, 0 slots)
# ============================================================
if MODE == 'train_hem_eq':
    print('\n[TRAIN] HEM hard-only equation_transform (lr=5e-6, 60 steps)')
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

    # Load datasets
    train_data, eval_data = load_holdout_eval()
    print(f'  Train (HOLDOUT excluded): {len(train_data)}')
    print(f'  Eval HOLDOUT: {len(eval_data)}')

    # HEM: filter equation_transform + cryptarithm hard
    hem_data = [r for r in train_data if r.get('_src') in ('eq', 'crypt')]
    print(f'  HEM (eq + crypt): {len(hem_data)}')

    from datasets import Dataset
    all_data = []
    for row in hem_data:
        p = (row.get('prompt') or '').strip()
        c = (row.get('cot') or row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c})
    ds = Dataset.from_list(all_data)
    print(f'  Train final: {len(ds)}')

    # Tokenize
    MAX_LENGTH = 4096
    def fmt(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['completion']}]
        full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_ids = tokenizer(full, add_special_tokens=False)['input_ids']
        prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']
        if len(full_ids) > MAX_LENGTH:
            prefix = min(len(prm_ids), MAX_LENGTH // 3)
            suffix = MAX_LENGTH - prefix
            full_ids = full_ids[:prefix] + full_ids[-suffix:]
        labels = list(full_ids)
        n_p = min(len(prm_ids), len(labels))
        for i in range(n_p):
            labels[i] = -100
        return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

    tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenize')

    PAD = tokenizer.pad_token_id or tokenizer.eos_token_id
    def coll(features):
        ml = max(len(f['input_ids']) for f in features)
        ml = ((ml + 7) // 8) * 8
        ids, ms, lb = [], [], []
        for f in features:
            p = ml - len(f['input_ids'])
            ids.append(f['input_ids'] + [PAD]*p)
            ms.append(f['attention_mask'] + [0]*p)
            lb.append(f['labels'] + [-100]*p)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(ms), 'labels': torch.tensor(lb)}

    from transformers import Trainer, TrainingArguments
    OUT = '/content/output_v159_train'
    os.makedirs(OUT, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=OUT,
        max_steps=60,                       # GPT-5.5 conservador
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=5e-6,                 # 5e-6 (Gemini Flash mais conservador)
        lr_scheduler_type='cosine',
        warmup_steps=6,
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.1,                   # Gemini Flash insight
        max_grad_norm=0.5,
        label_smoothing_factor=0.0,
        logging_steps=5, save_steps=20, save_total_limit=4,
        bf16=True, optim='adamw_torch_fused',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
        remove_unused_columns=False, report_to='none',
        dataloader_num_workers=0, seed=42,
        neftune_noise_alpha=5,
    )
    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=coll)
    print(f'  Steps: 60 | LR: 5e-6 cosine | wd: 0.1 | NEFTune: 5')
    t0 = time.time()
    trainer.train()
    print(f'\n[OK] Training: {(time.time()-t0)/60:.1f} min')

    AD_OUT = f'{OUT}/final_adapter'
    trainer.save_model(AD_OUT)
    tokenizer.save_pretrained(AD_OUT)

    try:
        GD = '/content/drive/My Drive/KG1_v159_trained_adapter'
        if os.path.exists(GD): shutil.rmtree(GD)
        shutil.copytree(AD_OUT, GD)
        print(f'  GDrive backup: {GD}')
    except Exception as e:
        print(f'  GDrive: {e}')

    print('\n=== TRAIN DONE - Run paired_validate_submit ===')
    raise SystemExit(0)

# ============================================================
# MODE 3: paired_validate_submit (30 min, 1 slot SE PASSAR)
# ============================================================
if MODE == 'paired_validate_submit':
    print('\n[PAIRED VALIDATE] vs V120 baseline em HOLDOUT')

    TRAINED = '/content/output_v159_train/final_adapter'
    if not os.path.exists(TRAINED):
        gd = '/content/drive/My Drive/KG1_v159_trained_adapter'
        if os.path.exists(gd):
            shutil.copytree(gd, TRAINED)
        else:
            raise RuntimeError('Trained adapter not found - run train_hem_eq first')

    model.load_adapter(TRAINED, adapter_name='trained')

    train_data, eval_data = load_holdout_eval()
    print(f'  HOLDOUT: {len(eval_data)} samples')

    print('\n  V120 baseline eval...')
    res_v = eval_adapter(eval_data, 'default', max_n=100)
    print(f'  V120: {res_v["correct"]}/{res_v["total"]}')

    print('\n  TRAINED eval...')
    res_t = eval_adapter(eval_data, 'trained', max_n=100)
    print(f'  TRAINED: {res_t["correct"]}/{res_t["total"]}')

    delta = res_t['correct'] - res_v['correct']
    print(f'\n  DELTA total: {delta:+d}')

    # NO-GO gate
    no_go = False
    print(f'\n  Per-dataset:')
    for src in ['crypt', 'eq', 'synth']:
        v = res_v['by_src'][src]; t = res_t['by_src'][src]
        if v['t'] == 0: continue
        d = t['c'] - v['c']
        marker = '❌ REGRESSION' if d < 0 else ('⚠️ tie' if d == 0 else '✅')
        print(f'    {src}: V={v["c"]}/{v["t"]} T={t["c"]}/{t["t"]} delta={d:+d} {marker}')
        if d < 0: no_go = True

    print(f'\n  === GATE ===')
    if no_go:
        print(f'  >>> ❌ ABORT: regressao detected')
        print(f'  >>> NEXT: try MODE format_only_minisft (fallback)')
    elif delta <= 0:
        print(f'  >>> ⚠️ no improvement, nao recomendado submit')
        no_go = True
    else:
        print(f'  >>> ✅ PASS: delta={delta:+d}, sem regressao')

    if not no_go and not DRY_RUN:
        SUBMIT = '/content/submit_v159_trained'
        os.makedirs(SUBMIT, exist_ok=True)
        for fn in ['adapter_config.json', 'adapter_model.safetensors']:
            src = os.path.join(TRAINED, fn)
            if os.path.exists(src): shutil.copy(src, SUBMIT)
        zip_path = '/content/submission.zip'  # NOME importante!
        if os.path.exists(zip_path): os.remove(zip_path)
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT}/*')], check=True)

        msg = f'v159 train_hem_eq lr=5e-6 60steps - paired delta={delta:+d}'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge',
                          '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=1800)
        print(f'\n  Submit: {r.stdout[:200]}')
        if r.stderr: print(f'  STDERR: {r.stderr[-200:]}')

    print('\n=== PAIRED VALIDATE DONE ===')
    raise SystemExit(0)

# ============================================================
# MODE 4: format_only_minisft (fallback - 30 min)
# ============================================================
if MODE == 'format_only_minisft':
    print('\n[FORMAT MINI-SFT] 100 hard samples, 10 steps lr=1e-6')
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

    # Find 100 samples where V120 fails format (Tipo C from prior diagnostic)
    train_data, eval_data = load_holdout_eval()

    # Run inference on first 200 train samples to find format issues
    print('  Finding 100 format-fail samples...')
    model.set_adapter('default')
    model.eval()
    fail_format_samples = []
    for row in train_data[:200]:
        if len(fail_format_samples) >= 100: break
        prompt = (row.get('prompt') or '').strip()
        gold = get_gold(row)
        if not prompt or not gold: continue
        msgs = [{'role': 'user', 'content': prompt + PROMPT_SUFFIX}]
        inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                                add_generation_prompt=True, enable_thinking=True).to('cuda')
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=1500, temperature=0.0, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        ms = KAGGLE_REGEX.findall(text)
        ext = next((m.strip() for m in reversed(ms) if m.strip()), None) if ms else None
        # If has oxed but format off (gold present in text but parser fails)
        if ext and not kaggle_verify(gold, ext) and gold.lower() in text.lower():
            # Format issue - good candidate
            fail_format_samples.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': row.get('cot', '') + f'\n\\boxed{{{gold}}}'  # FORCE strict format
            })

    print(f'  Found {len(fail_format_samples)} format-fail samples')
    if len(fail_format_samples) < 30:
        print(f'  WARN: poucos format-fail samples ({len(fail_format_samples)} < 30)')
        print(f'  Recomendacao: Skip format_only, use model_soup ou variance hunt')
        raise SystemExit(0)

    # Train 10 steps lr=1e-6
    from datasets import Dataset
    ds = Dataset.from_list(fail_format_samples)

    MAX_LENGTH = 4096
    def fmt(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['completion']}]
        full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_ids = tokenizer(full, add_special_tokens=False)['input_ids'][:MAX_LENGTH]
        prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids'][:MAX_LENGTH]
        labels = list(full_ids)
        for i in range(min(len(prm_ids), len(labels))): labels[i] = -100
        return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

    tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=2, desc='Tokenize format')

    PAD = tokenizer.pad_token_id or tokenizer.eos_token_id
    def coll(features):
        ml = max(len(f['input_ids']) for f in features)
        ml = ((ml + 7) // 8) * 8
        ids, ms_, lb = [], [], []
        for f in features:
            p = ml - len(f['input_ids'])
            ids.append(f['input_ids'] + [PAD]*p)
            ms_.append(f['attention_mask'] + [0]*p)
            lb.append(f['labels'] + [-100]*p)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(ms_), 'labels': torch.tensor(lb)}

    from transformers import Trainer, TrainingArguments
    OUT = '/content/output_v159_format'
    os.makedirs(OUT, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=OUT,
        max_steps=10, per_device_train_batch_size=1, gradient_accumulation_steps=10,
        learning_rate=1e-6, lr_scheduler_type='cosine', warmup_steps=2,
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.0, max_grad_norm=0.5, label_smoothing_factor=0.0,
        logging_steps=2, save_steps=5, save_total_limit=2,
        bf16=True, optim='adamw_torch_fused',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
        remove_unused_columns=False, report_to='none',
        dataloader_num_workers=0, seed=42,
    )
    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=coll)
    print(f'  Steps: 10 | LR: 1e-6 | Format-strict')
    trainer.train()

    AD_OUT = f'{OUT}/final_adapter'
    trainer.save_model(AD_OUT)
    tokenizer.save_pretrained(AD_OUT)
    print(f'  Saved: {AD_OUT}')

    # Auto-submit
    if not DRY_RUN:
        SUBMIT = '/content/submit_v159_format'
        os.makedirs(SUBMIT, exist_ok=True)
        for fn in ['adapter_config.json', 'adapter_model.safetensors']:
            src = os.path.join(AD_OUT, fn)
            if os.path.exists(src): shutil.copy(src, SUBMIT)
        zip_path = '/content/submission.zip'
        if os.path.exists(zip_path): os.remove(zip_path)
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT}/*')], check=True)

        msg = f'v159 format_only_minisft 100 samples 10 steps lr=1e-6'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge',
                          '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=1800)
        print(f'  Submit: {r.stdout[:200]}{r.stderr[-200:]}')

    print('\n=== FORMAT MINI-SFT DONE ===')
    raise SystemExit(0)

# ============================================================
# MODE 5: model_soup (fallback - 30 min)
# ============================================================
if MODE == 'model_soup':
    print('\n[MODEL SOUP] Average V120 + V121 weights (50/50)')

    TRAINED = '/content/output_v159_train/final_adapter'
    if not os.path.exists(TRAINED):
        gd = '/content/drive/My Drive/KG1_v159_trained_adapter'
        if os.path.exists(gd):
            shutil.copytree(gd, TRAINED)
        else:
            raise RuntimeError('V121 (trained) not found')

    # Load V120 + V121 safetensors and average
    from safetensors.torch import load_file, save_file
    print('  Loading V120 weights...')
    v120 = load_file(os.path.join(adapter_dir, 'adapter_model.safetensors'))
    print('  Loading V121 weights...')
    v121 = load_file(os.path.join(TRAINED, 'adapter_model.safetensors'))

    print('  Averaging (50/50)...')
    soup = {}
    for k in v120.keys():
        if k in v121 and v120[k].shape == v121[k].shape:
            soup[k] = (v120[k].float() + v121[k].float()) / 2.0
            soup[k] = soup[k].to(v120[k].dtype)
        else:
            soup[k] = v120[k]
            print(f'    [skip] {k}: shape mismatch or missing in V121')

    SOUP_DIR = '/content/output_v159_soup'
    os.makedirs(SOUP_DIR, exist_ok=True)
    save_file(soup, os.path.join(SOUP_DIR, 'adapter_model.safetensors'))
    shutil.copy(os.path.join(adapter_dir, 'adapter_config.json'), SOUP_DIR)
    print(f'  Soup saved: {SOUP_DIR}')

    # Auto-submit
    if not DRY_RUN:
        SUBMIT = '/content/submit_v159_soup'
        os.makedirs(SUBMIT, exist_ok=True)
        for fn in ['adapter_config.json', 'adapter_model.safetensors']:
            shutil.copy(os.path.join(SOUP_DIR, fn), SUBMIT)
        zip_path = '/content/submission.zip'
        if os.path.exists(zip_path): os.remove(zip_path)
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT}/*')], check=True)

        msg = f'v159 model_soup V120+V121 50-50 average'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge',
                          '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=1800)
        print(f'  Submit: {r.stdout[:200]}')

    print('\n=== MODEL SOUP DONE ===')
    raise SystemExit(0)

print(f'\n[ERROR] Unknown MODE: {MODE}')
